# Task A Notebook v0 — On-Demand User Profiles + Rating/Review Simulation

Goal for today:
1. Load the prepared train/internal validation splits.
2. Build a compact user evidence packet from train history only.
3. Generate a user profile on demand using Gemini or DeepSeek.
4. Cache the profile.
5. Predict rating + generate review for a held-out item.

This notebook is intentionally simple and iteration-friendly. Once the prompts/methods are stable, we can move the code into clean scripts for the app/container.

In [1]:
# Optional, only run if needed:
# %pip install pandas pyarrow requests tqdm

import os
import json
import math
import time
import hashlib
import re
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import pandas as pd
import requests
from tqdm.auto import tqdm

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 160)

In [2]:
# -------------------------
# Config
# -------------------------

OUTPUT_DIR = Path("data/processed/baseline_v0")
SPLITS_DIR = OUTPUT_DIR / "splits"
CACHE_DIR = OUTPUT_DIR / "cache"
PROFILE_CACHE_DIR = CACHE_DIR / "user_profiles"
PROFILE_CACHE_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = SPLITS_DIR / "train.parquet"
INTERNAL_VAL_PATH = SPLITS_DIR / "internal_val.parquet"
ITEMS_PATH = OUTPUT_DIR / "items.parquet"  # exists only if you did not run --skip-meta

# Switch this as needed.
LLM_PROVIDER = os.getenv("LLM_PROVIDER", "gemini")  # "gemini" or "deepseek"
GEMINI_MODEL = os.getenv("GEMINI_MODEL", "gemini-2.5-flash-lite")

# Use whichever DeepSeek model your account supports.
# If "deepseek-v4-flash" works for you, keep it.
# Otherwise try "deepseek-v4-pro".
DEEPSEEK_MODEL = os.getenv("DEEPSEEK_MODEL", "deepseek-v4-flash")

# Keep low while evaluating. Higher temperature causes more JSON mistakes.
TEMPERATURE = 0.1

# Give enough room so JSON does not get cut off mid-response.
MAX_OUTPUT_TOKENS = 2200

# Retry failed JSON/schema calls.
LLM_RETRIES = 2

print("Provider:", LLM_PROVIDER)
print("Train exists:", TRAIN_PATH.exists())
print("Internal val exists:", INTERNAL_VAL_PATH.exists())
print("Items exists:", ITEMS_PATH.exists())

Provider: gemini
Train exists: True
Internal val exists: True
Items exists: True


In [4]:
from pathlib import Path
import pandas as pd
import json

TASK_A_DIR = Path("data/processed/task_a_iteration_v0")

train_df = pd.read_parquet(TASK_A_DIR / "train.parquet")
internal_val_df = pd.read_parquet(TASK_A_DIR / "internal_val.parquet")
shadow_test_df = pd.read_parquet(TASK_A_DIR / "shadow_test.parquet")
items_df = pd.read_parquet(TASK_A_DIR / "items.parquet")

with open(TASK_A_DIR / "summary.json", "r", encoding="utf-8") as f:
    saved_summary = json.load(f)

item_lookup = {
    (row["domain"], row["parent_asin"]): row.to_dict()
    for _, row in items_df.iterrows()
}

print("Loaded filtered Task A iteration data:")
print("train:", train_df.shape)
print("internal_val:", internal_val_df.shape)
print("shadow_test:", shadow_test_df.shape)
print("items:", items_df.shape)

saved_summary

Loaded filtered Task A iteration data:
train: (190457, 13)
internal_val: (20408, 13)
shadow_test: (17348, 13)
items: (50927, 11)


{'train_shape': [190457, 13],
 'internal_val_shape': [20408, 13],
 'shadow_test_shape': [17348, 13],
 'items_shape': [50927, 11],
 'train_users': 5978,
 'internal_val_users': 5978,
 'shadow_test_users': 4975,
 'train_domain_counts': {'Movies_and_TV': 104696, 'Books': 85761},
 'internal_val_domain_counts': {'Movies_and_TV': 11570, 'Books': 8838},
 'shadow_test_domain_counts': {'Movies_and_TV': 9848, 'Books': 7500},
 'n_items': 50927}

In [50]:
# -------------------------
# Small helpers
# -------------------------

def clean_text(x: Any, max_chars: Optional[int] = None) -> str:
    if x is None or (isinstance(x, float) and math.isnan(x)):
        s = ""
    elif isinstance(x, list):
        s = " ".join(clean_text(v) for v in x)
    elif isinstance(x, dict):
        s = json.dumps(x, ensure_ascii=False)
    else:
        s = str(x)

    # Collapse whitespace so raw newlines do not pollute prompts.
    s = re.sub(r"\s+", " ", s).strip()

    if max_chars and len(s) > max_chars:
        return s[:max_chars].rsplit(" ", 1)[0] + "..."
    return s


def safe_json_loads(text: str) -> Dict[str, Any]:
    """
    Parse JSON even when the model wraps it in markdown or extra text.

    This does not magically repair broken JSON. It only:
    - strips markdown fences
    - extracts the first {...} block
    - parses it with json.loads
    """
    if text is None:
        raise ValueError("LLM returned empty response.")

    text = str(text).strip()

    # Remove markdown fences.
    text = re.sub(r"^```(?:json)?\s*", "", text, flags=re.IGNORECASE).strip()
    text = re.sub(r"\s*```$", "", text).strip()

    try:
        return json.loads(text)
    except json.JSONDecodeError:
        start = text.find("{")
        end = text.rfind("}")
        if start != -1 and end != -1 and end > start:
            return json.loads(text[start:end + 1])
        raise


def rating_distribution(series: pd.Series) -> Dict[str, int]:
    counts = series.round().astype(int).value_counts().to_dict()
    return {str(k): int(counts.get(k, 0)) for k in range(1, 6)}


def summarize_item(domain: str, parent_asin: str, max_chars: int = 700) -> Dict[str, Any]:
    meta = item_lookup.get((str(domain), str(parent_asin)), {})
    return {
        "domain": str(domain),
        "parent_asin": str(parent_asin),
        "title": clean_text(meta.get("title", ""), 180),
        "store": clean_text(meta.get("store", ""), 80),
        "categories": clean_text(meta.get("categories", ""), 250),
        "features": clean_text(meta.get("features", ""), max_chars),
        "description": clean_text(meta.get("description", ""), max_chars),
        "price": meta.get("price", None),
    }


def compact_review_row(row: pd.Series, include_item_meta: bool = True) -> Dict[str, Any]:
    item = summarize_item(row["domain"], row["parent_asin"], max_chars=300) if include_item_meta else {}
    return {
        "domain": row["domain"],
        "parent_asin": row["parent_asin"],
        "rating": float(row["rating"]),
        "review_title": clean_text(row.get("review_title", ""), 120),
        "review_text": clean_text(row.get("review_text", ""), 700),
        "timestamp": int(row["timestamp"]),
        "verified_purchase": bool(row.get("verified_purchase", False)),
        "helpful_vote": int(row.get("helpful_vote", 0)),
        "item_title": item.get("title", ""),
        "item_categories": item.get("categories", ""),
    }


# -------------------------
# JSON schema / validation helpers
# -------------------------

TASK_A_REQUIRED_KEYS = {
    "predicted_rating",
    "rating_confidence",
    "rating_rationale",
    "generated_review_title",
    "generated_review_text",
    "style_match_notes",
    "possible_failure_modes",
}

PROFILE_REQUIRED_KEYS = {
    "user_summary",
    "rating_behavior",
    "domain_preferences",
    "review_style",
    "useful_prediction_cues",
    "uncertainties",
}


def _as_clean_string(value: Any, max_chars: Optional[int] = None) -> str:
    """Force any value into a safe single-line string."""
    s = clean_text(value, max_chars=max_chars)
    return s


def _as_string_list(value: Any, fallback: Optional[List[str]] = None) -> List[str]:
    """Force a value into list[str]."""
    if fallback is None:
        fallback = []

    if value is None:
        return fallback

    if isinstance(value, list):
        out = [_as_clean_string(v, max_chars=300) for v in value]
        return [v for v in out if v]

    if isinstance(value, str):
        value = clean_text(value)
        return [value] if value else fallback

    return [_as_clean_string(value, max_chars=300)]


def validate_task_a_prediction(obj: Dict[str, Any]) -> Dict[str, Any]:
    """
    Validate and normalize Task A JSON.

    Raises ValueError if required structure is missing.
    Also clamps rating to [1.0, 5.0].
    """
    if not isinstance(obj, dict):
        raise ValueError(f"Task A prediction must be a JSON object. Got: {type(obj)}")

    missing = TASK_A_REQUIRED_KEYS - set(obj.keys())
    if missing:
        raise ValueError(f"Task A prediction missing keys: {sorted(missing)}")

    # Rating.
    try:
        rating = float(obj["predicted_rating"])
        rating = max(1.0, min(5.0, rating))
    except Exception as e:
        raise ValueError(f"Invalid predicted_rating={obj.get('predicted_rating')!r}: {repr(e)}")

    # Confidence.
    confidence = clean_text(obj.get("rating_confidence", "")).lower()
    if confidence not in {"low", "medium", "high"}:
        raise ValueError(f"Invalid rating_confidence={obj.get('rating_confidence')!r}")

    cleaned = {
        "predicted_rating": rating,
        "rating_confidence": confidence,
        "rating_rationale": _as_clean_string(obj.get("rating_rationale", ""), max_chars=600),
        "generated_review_title": _as_clean_string(obj.get("generated_review_title", ""), max_chars=180),
        "generated_review_text": _as_clean_string(obj.get("generated_review_text", ""), max_chars=1800),
        "style_match_notes": _as_string_list(obj.get("style_match_notes"), fallback=[]),
        "possible_failure_modes": _as_string_list(obj.get("possible_failure_modes"), fallback=[]),
    }

    if not cleaned["generated_review_text"]:
        raise ValueError("generated_review_text is empty.")

    return cleaned


def validate_user_profile(obj: Dict[str, Any]) -> Dict[str, Any]:
    """
    Validate and lightly normalize user-profile JSON.

    This is intentionally less strict than Task A because profile text can vary.
    """
    if not isinstance(obj, dict):
        raise ValueError(f"User profile must be a JSON object. Got: {type(obj)}")

    missing = PROFILE_REQUIRED_KEYS - set(obj.keys())
    if missing:
        raise ValueError(f"User profile missing keys: {sorted(missing)}")

    # Normalize top-level text/list fields.
    obj["user_summary"] = _as_clean_string(obj.get("user_summary", ""), max_chars=1000)
    obj["useful_prediction_cues"] = _as_string_list(obj.get("useful_prediction_cues"), fallback=[])
    obj["uncertainties"] = _as_string_list(obj.get("uncertainties"), fallback=[])

    # Ensure nested sections are dicts.
    for key in ["rating_behavior", "domain_preferences", "review_style"]:
        if not isinstance(obj.get(key), dict):
            raise ValueError(f"{key} must be a JSON object.")

    return obj

In [35]:
def build_user_evidence_packet(
    user_id: str,
    target_domain: Optional[str] = None,
    target_parent_asin: Optional[str] = None,
    train: pd.DataFrame = train_df,
    max_examples: int = 12,
    n_recent: int = 4,
    n_high: int = 3,
    n_low: int = 3,
    n_same_domain: int = 4,
) -> Dict[str, Any]:
    """
    Build a compact, train-only evidence packet for one user.

    Selection strategy:
    - rating stats and domain counts
    - recent reviews
    - high-rating examples
    - low-rating examples
    - same-domain examples for the target domain
    Deduplicates examples and caps prompt size.
    """
    user_hist = train[train["user_id"].astype(str) == str(user_id)].copy()
    if user_hist.empty:
        raise ValueError(f"No train history found for user_id={user_id}")

    user_hist = user_hist.sort_values(["timestamp", "domain", "parent_asin"]).reset_index(drop=True)
    ratings = user_hist["rating"].astype(float)

    # Basic style stats.
    text_lengths = user_hist["review_text"].fillna("").map(lambda x: len(str(x).split()))
    title_lengths = user_hist["review_title"].fillna("").map(lambda x: len(str(x).split()))

    domain_counts = user_hist["domain"].value_counts().to_dict()
    domain_avg_ratings = user_hist.groupby("domain")["rating"].mean().round(3).to_dict()

    # Candidate review groups.
    recent = user_hist.tail(n_recent)
    high = user_hist.sort_values(["rating", "timestamp"], ascending=[False, False]).head(n_high)
    low = user_hist.sort_values(["rating", "timestamp"], ascending=[True, False]).head(n_low)

    groups = [("recent", recent), ("high_rating", high), ("low_rating", low)]

    if target_domain:
        same_domain = user_hist[user_hist["domain"] == target_domain].tail(n_same_domain)
        if not same_domain.empty:
            groups.append(("same_domain", same_domain))

    # Deduplicate by interaction identity.
    seen = set()
    examples = []
    for reason, df in groups:
        for _, row in df.iterrows():
            key = (row["domain"], row["parent_asin"], int(row["timestamp"]), float(row["rating"]))
            if key in seen:
                continue
            seen.add(key)
            ex = compact_review_row(row)
            ex["selection_reason"] = reason
            examples.append(ex)
            if len(examples) >= max_examples:
                break
        if len(examples) >= max_examples:
            break

    packet = {
        "user_id": str(user_id),
        "history_scope": "train_only",
        "n_train_reviews": int(len(user_hist)),
        "domains_seen": sorted([str(x) for x in user_hist["domain"].unique()]),
        "domain_counts": {str(k): int(v) for k, v in domain_counts.items()},
        "domain_avg_ratings": {str(k): float(v) for k, v in domain_avg_ratings.items()},
        "rating_stats": {
            "mean": float(ratings.mean()),
            "median": float(ratings.median()),
            "min": float(ratings.min()),
            "max": float(ratings.max()),
            "std": float(ratings.std()) if len(ratings) > 1 else 0.0,
            "distribution": rating_distribution(ratings),
        },
        "review_style_stats": {
            "avg_review_words": float(text_lengths.mean()),
            "median_review_words": float(text_lengths.median()),
            "avg_title_words": float(title_lengths.mean()),
            "empty_review_frac": float((text_lengths == 0).mean()),
            "verified_purchase_frac": float(user_hist["verified_purchase"].mean()) if "verified_purchase" in user_hist else None,
        },
        # "target_context": {
        #     "target_domain": target_domain,
        #     "target_parent_asin": target_parent_asin,
        # },
        "selected_review_examples": examples,
    }
    return packet

# Quick smoke test on one validation row.
sample_row = internal_val_df.sample(1, random_state=42).iloc[0]
packet = build_user_evidence_packet(
    user_id=sample_row["user_id"],
    # target_domain=sample_row["domain"],
    # target_parent_asin=sample_row["parent_asin"],
)
packet["n_train_reviews"], packet["rating_stats"], len(packet["selected_review_examples"])

(48,
 {'mean': 4.666666666666667,
  'median': 5.0,
  'min': 3.0,
  'max': 5.0,
  'std': 0.6302087912488811,
  'distribution': {'1': 0, '2': 0, '3': 4, '4': 8, '5': 36}},
 7)

In [51]:
def _raw_llm_call(
    prompt: str,
    provider: str = LLM_PROVIDER,
    model: Optional[str] = None,
    temperature: float = TEMPERATURE,
    max_output_tokens: int = MAX_OUTPUT_TOKENS,
    json_mode: bool = True,
    timeout: int = 90,
) -> Dict[str, Any]:
    """
    One raw provider call. No retry and no schema validation here.

    Returns:
      {"raw_text": "...", "provider": "...", "model": "..."}
    """
    provider = provider.lower().strip()

    if provider == "gemini":
        api_key = os.getenv("GEMINI_API_KEY")
        if not api_key:
            raise RuntimeError("Missing GEMINI_API_KEY environment variable.")

        model = model or GEMINI_MODEL
        url = f"https://generativelanguage.googleapis.com/v1beta/models/{model}:generateContent"
        params = {"key": api_key}

        payload = {
            "contents": [
                {
                    "role": "user",
                    "parts": [{"text": prompt}],
                }
            ],
            "generationConfig": {
                "temperature": temperature,
                "maxOutputTokens": max_output_tokens,
            },
        }

        if json_mode:
            payload["generationConfig"]["responseMimeType"] = "application/json"

        resp = requests.post(url, params=params, json=payload, timeout=timeout)
        resp.raise_for_status()
        data = resp.json()

        try:
            raw_text = data["candidates"][0]["content"]["parts"][0]["text"]
        except Exception as e:
            raise RuntimeError(f"Unexpected Gemini response shape: {data}") from e

    elif provider == "deepseek":
        api_key = os.getenv("DEEPSEEK_API_KEY")
        if not api_key:
            raise RuntimeError("Missing DEEPSEEK_API_KEY environment variable.")

        model = model or DEEPSEEK_MODEL
        url = "https://api.deepseek.com/chat/completions"

        headers = {
            "Authorization": f"Bearer {api_key}",
            "Content-Type": "application/json",
        }

        payload = {
            "model": model,
            "messages": [
                {
                    "role": "system",
                    "content": (
                        "You are a precise recommendation and user-modelling assistant. "
                        "Return exactly one valid JSON object. "
                        "Do not use markdown. Do not include prose outside JSON. "
                        "Do not use comments, trailing commas, NaN, Infinity, or undefined."
                    ),
                },
                {
                    "role": "user",
                    "content": prompt,
                },
            ],
            "temperature": temperature,
            "max_tokens": max_output_tokens,
            "stream": False,
        }

        if json_mode:
            # DeepSeek JSON mode: valid JSON object, but not strict schema enforcement.
            payload["response_format"] = {"type": "json_object"}

        resp = requests.post(url, headers=headers, json=payload, timeout=timeout)
        resp.raise_for_status()
        data = resp.json()

        try:
            raw_text = data["choices"][0]["message"]["content"]
        except Exception as e:
            raise RuntimeError(f"Unexpected DeepSeek response shape: {data}") from e

    else:
        raise ValueError("provider must be 'gemini' or 'deepseek'")

    return {
        "provider": provider,
        "model": model,
        "raw_text": raw_text,
    }


def make_json_repair_prompt(
    bad_raw_text: str,
    error: Exception,
    schema_name: str,
) -> str:
    """
    Ask the model to repair only its own broken JSON.
    This is cheaper and safer than rerunning the full task every time.
    """
    if schema_name == "task_a":
        schema = """
{
  "predicted_rating": 4.0,
  "rating_confidence": "medium",
  "rating_rationale": "string",
  "generated_review_title": "string",
  "generated_review_text": "string",
  "style_match_notes": ["string"],
  "possible_failure_modes": ["string"]
}
""".strip()
    elif schema_name == "profile":
        schema = """
{
  "user_summary": "string",
  "rating_behavior": {
    "average_rating_interpretation": "string",
    "rating_scale_harshness": "lenient",
    "what_gets_5_stars": "string",
    "what_gets_low_ratings": "string",
    "calibration_rule": "string"
  },
  "domain_preferences": {
    "Books": "string",
    "Movies_and_TV": "string",
    "cross_domain": "string",
    "Genres": "string"
  },
  "review_style": {
    "typical_length": "medium",
    "tone": "string",
    "specificity": "medium",
    "common_focus": ["string"],
    "title_style": "string",
    "common_words": "string"
  },
  "useful_prediction_cues": ["string"],
  "uncertainties": ["string"]
}
""".strip()
    else:
        schema = "{ }"

    return f"""
Your previous response was not valid JSON or did not match the required schema.

Error:
{repr(error)}

Repair the response below.

Rules:
- Return exactly one valid JSON object.
- No markdown fences.
- No explanation outside JSON.
- No comments.
- No trailing commas.
- No NaN, Infinity, null, or undefined.
- Escape quotes and newlines inside strings correctly.
- Use this schema shape:

{schema}

Broken response:
{bad_raw_text}
""".strip()


def call_llm(
    prompt: str,
    provider: str = LLM_PROVIDER,
    model: Optional[str] = None,
    temperature: float = TEMPERATURE,
    max_output_tokens: int = MAX_OUTPUT_TOKENS,
    json_mode: bool = True,
    timeout: int = 90,
    schema_name: Optional[str] = None,   # None, "task_a", or "profile"
    retries: int = LLM_RETRIES,
) -> Dict[str, Any]:
    """
    Provider-switchable LLM call with:
    - JSON parsing
    - optional schema validation
    - retry/repair loop

    Returns:
      {
        "raw_text": "...",
        "json": {...} or None,
        "provider": "...",
        "model": "...",
        "attempts": int,
        "repaired": bool
      }
    """
    last_error = None
    last_raw_text = None
    repaired = False

    current_prompt = prompt

    for attempt in range(retries + 1):
        result = _raw_llm_call(
            prompt=current_prompt,
            provider=provider,
            model=model,
            temperature=temperature,
            max_output_tokens=max_output_tokens,
            json_mode=json_mode,
            timeout=timeout,
        )

        raw_text = result["raw_text"]
        last_raw_text = raw_text

        try:
            parsed = None

            if json_mode:
                parsed = safe_json_loads(raw_text)

                if schema_name == "task_a":
                    parsed = validate_task_a_prediction(parsed)
                elif schema_name == "profile":
                    parsed = validate_user_profile(parsed)

            return {
                "provider": result["provider"],
                "model": result["model"],
                "raw_text": raw_text,
                "json": parsed,
                "attempts": attempt + 1,
                "repaired": repaired,
            }

        except Exception as e:
            last_error = e

            # Next attempt: repair the broken JSON only.
            current_prompt = make_json_repair_prompt(
                bad_raw_text=raw_text,
                error=e,
                schema_name=schema_name or "generic",
            )
            repaired = True

            # Keep repair deterministic.
            temperature = 0.0

    raise ValueError(
        "LLM JSON/schema parsing failed after "
        f"{retries + 1} attempt(s). "
        f"Last error={repr(last_error)}. "
        f"Last raw response preview={repr((last_raw_text or '')[:2500])}"
    )

In [52]:
PROFILE_PROMPT_VERSION = "user_profile_v2_strict_json"

def make_user_profile_prompt(evidence_packet: Dict[str, Any]) -> str:
    return f"""
You are building a compact behavioural profile for a recommender-system user.

Use ONLY the evidence packet provided below.
Do not invent demographics, nationality, age, gender, location, occupation, or life context.

The profile should help another model predict:
1. the user's likely star rating for unseen Books or Movies_and_TV items
2. the user's review tone, length, and writing style
3. transferable preferences across Books and Movies_and_TV

JSON rules:
- Return exactly one valid JSON object and nothing else.
- Do not use markdown fences.
- Do not include comments.
- Do not include trailing commas.
- Do not use NaN, Infinity, null, or undefined.
- All strings must escape internal quotes and newlines correctly.
- rating_scale_harshness must be exactly one of: "lenient", "moderate", "harsh", "unknown".
- typical_length must be exactly one of: "short", "medium", "long", "mixed".
- specificity must be exactly one of: "low", "medium", "high".

Return this exact JSON structure:
{{
  "user_summary": "2-4 sentence summary of the user's preference pattern.",
  "rating_behavior": {{
    "average_rating_interpretation": "How to interpret the user's average rating.",
    "rating_scale_harshness": "moderate",
    "what_gets_5_stars": "Patterns that appear to earn 5 stars from this user.",
    "what_gets_low_ratings": "Patterns that appear to earn low ratings from this user.",
    "calibration_rule": "How to anchor future predictions to this user's mean and rating distribution."
  }},
  "domain_preferences": {{
    "Books": "Book-specific preferences supported by evidence.",
    "Movies_and_TV": "Movie/TV-specific preferences supported by evidence.",
    "cross_domain": "Domain-agnostic preferences that transfer between books and movies.",
    "Genres": "Likely genres or content types, only if supported by evidence."
  }},
  "review_style": {{
    "typical_length": "medium",
    "tone": "The user's likely tone.",
    "specificity": "medium",
    "common_focus": ["Aspects the user often comments on."],
    "title_style": "How this user tends to title reviews.",
    "common_words": "Repeated words or phrasing patterns, if visible."
  }},
  "useful_prediction_cues": ["Concrete cues that should influence future rating/review predictions."],
  "uncertainties": ["Important limitations or missing evidence."]
}}

Evidence packet:
{json.dumps(evidence_packet, ensure_ascii=False)}
""".strip()


def profile_cache_path(user_id: str, provider: str = LLM_PROVIDER, model: Optional[str] = None) -> Path:
    model = model or (GEMINI_MODEL if provider == "gemini" else DEEPSEEK_MODEL)
    key = f"{PROFILE_PROMPT_VERSION}|{provider}|{model}|{user_id}"
    h = hashlib.sha1(key.encode("utf-8")).hexdigest()
    return PROFILE_CACHE_DIR / f"{h}.json"


def generate_user_profile(
    user_id: str,
    target_domain: Optional[str] = None,
    target_parent_asin: Optional[str] = None,
    provider: str = LLM_PROVIDER,
    model: Optional[str] = None,
    force_refresh: bool = False,
) -> Dict[str, Any]:
    cache_path = profile_cache_path(user_id, provider=provider, model=model)

    if cache_path.exists() and not force_refresh:
        return json.loads(cache_path.read_text())

    evidence_packet = build_user_evidence_packet(
        user_id=user_id,
        target_domain=target_domain,
        target_parent_asin=target_parent_asin,
    )

    prompt = make_user_profile_prompt(evidence_packet)

    result = call_llm(
        prompt=prompt,
        provider=provider,
        model=model,
        json_mode=True,
        max_output_tokens=1800,
        temperature=0.1,
        schema_name="profile",
        retries=LLM_RETRIES,
    )

    payload = {
        "user_id": str(user_id),
        "provider": result["provider"],
        "model": result["model"],
        "prompt_version": PROFILE_PROMPT_VERSION,
        "created_at": time.time(),
        "attempts": result.get("attempts"),
        "repaired": result.get("repaired"),
        "evidence_packet": evidence_packet,
        "profile": result["json"],
    }

    cache_path.write_text(json.dumps(payload, indent=2, ensure_ascii=False))
    return payload

In [47]:
import os
from getpass import getpass

# os.environ["GEMINI_API_KEY"] = getpass("Enter Gemini API key: ")
os.environ["DEEPSEEK_API_KEY"] = getpass("Enter DeepSeek API key: ")
# os.environ["LLM_PROVIDER"] = "deepseek"
# os.environ["LLM_PROVIDER"] = "gemini"

In [37]:
# Test profile generation.


profile_payload = generate_user_profile(
    user_id=sample_row["user_id"],
    provider="gemini",   # or "deepseek",
    model = "gemini-2.5-flash-lite",
    force_refresh=False,
)
print(json.dumps(profile_payload["profile"], indent=2, ensure_ascii=False))

{
  "user_summary": "This user has a strong preference for books, consistently rating them highly with an average of 4.67 stars. They enjoy books with elements of adventure, suspense, romance, and strong characters, though they express a dislike for excessive sex and profanity. Their reviews are typically medium-length, positive in tone, and focus on plot elements and character dynamics.",
  "rating_behavior": {
    "average_rating_interpretation": "The user is a very positive rater, with a high average rating and a strong tendency to give 5-star reviews.",
    "rating_scale_harshness": "lenient",
    "what_gets_5_stars": "Books with adventure, suspense, romance, strong characters, and humor. They appreciate depth in characters and enjoyable plots.",
    "what_gets_low_ratings": "Books that are too heavy or dark, or have an over-abundance of sex and profanity. Length can also be a minor negative factor if it's longer than preferred.",
    "calibration_rule": "Anchor future predictions 

In [53]:
TASK_A_PROMPT_VERSION = "task_a_rating_review_v2_strict_json"

def make_task_a_prompt(
    user_profile: Dict[str, Any],
    evidence_packet: Dict[str, Any],
    target_item: Dict[str, Any],
    locale_hint: Optional[str] = None,
) -> str:
    locale_instruction = ""
    if locale_hint:
        locale_instruction = f"""
Locale/context instruction:
The user context includes: {locale_hint}
Reflect this only if it is natural for the review style.
Avoid caricature, forced slang, exaggerated pidgin, or stereotypes.
""".strip()

    return f"""
You are simulating how a specific Amazon reviewer would rate and review an unseen item.

Use ONLY:
- the user's behavioural profile
- the user's selected past reviews
- the user's rating statistics
- the target item metadata

Core task:
1. Predict the star rating first.
2. Anchor the rating to the user's rating distribution and calibration rule.
3. Then generate a review title and review text that match the predicted rating.
4. Match the user's usual review length, tone, specificity, and title style.

Important behavioural rules:
- Do not predict only based on item popularity.
- Do not make every prediction positive.
- If the user's history is mostly generous, reflect that.
- If the user's history is harsh or selective, reflect that.
- If target metadata is weak, rely more on user rating behaviour and state uncertainty in possible_failure_modes.
- Do not mention facts not supported by the target item metadata.
- Do not copy any previous review text.
- Do not mention that you are an AI, model, simulator, or recommender system.

JSON rules:
- Return exactly one valid JSON object and nothing else.
- Do not use markdown fences.
- Do not include comments.
- Do not include trailing commas.
- Do not use NaN, Infinity, null, or undefined.
- All string values must escape internal quotes and newlines correctly.
- predicted_rating must be a number between 1.0 and 5.0, not a string.
- rating_confidence must be exactly one of: "low", "medium", "high".
- style_match_notes must be an array of strings.
- possible_failure_modes must be an array of strings.

Return this exact JSON structure:
{{
  "predicted_rating": 4.0,
  "rating_confidence": "medium",
  "rating_rationale": "Brief reason anchored to the user's profile, rating stats, selected evidence, and target item fit.",
  "generated_review_title": "Short title in the user's likely style.",
  "generated_review_text": "Review text in the user's likely style.",
  "style_match_notes": ["Short note about tone, length, specificity, or title style."],
  "possible_failure_modes": ["Short note about uncertainty, sparse metadata, or weak evidence."]
}}

{locale_instruction}

User profile:
{json.dumps(user_profile, ensure_ascii=False)}

Selected past-review evidence:
{json.dumps(evidence_packet["selected_review_examples"], ensure_ascii=False)}

User rating stats:
{json.dumps(evidence_packet["rating_stats"], ensure_ascii=False)}

Target item:
{json.dumps(target_item, ensure_ascii=False)}
""".strip()


def predict_task_a(
    user_id: str,
    target_domain: str,
    target_parent_asin: str,
    provider: str = LLM_PROVIDER,
    model: Optional[str] = None,
    locale_hint: Optional[str] = None,
    force_profile_refresh: bool = False,
) -> Dict[str, Any]:
    evidence_packet = build_user_evidence_packet(
        user_id=user_id,
        target_domain=target_domain,
        target_parent_asin=target_parent_asin,
    )

    profile_payload = generate_user_profile(
        user_id=user_id,
        target_domain=target_domain,
        target_parent_asin=target_parent_asin,
        provider=provider,
        model=model,
        force_refresh=force_profile_refresh,
    )

    target_item = summarize_item(target_domain, target_parent_asin, max_chars=900)

    prompt = make_task_a_prompt(
        user_profile=profile_payload["profile"],
        evidence_packet=evidence_packet,
        target_item=target_item,
        locale_hint=locale_hint,
    )

    result = call_llm(
        prompt=prompt,
        provider=provider,
        model=model,
        json_mode=True,
        max_output_tokens=1600,
        temperature=0.1,
        schema_name="task_a",
        retries=LLM_RETRIES,
    )

    out = result["json"]

    return {
        "user_id": str(user_id),
        "target_domain": str(target_domain),
        "target_parent_asin": str(target_parent_asin),
        "provider": result["provider"],
        "model": result["model"],
        "attempts": result.get("attempts"),
        "repaired": result.get("repaired"),
        "profile_prompt_version": PROFILE_PROMPT_VERSION,
        "task_a_prompt_version": TASK_A_PROMPT_VERSION,
        "target_item": target_item,
        "profile": profile_payload["profile"],
        "prediction": out,
        "raw_text": result["raw_text"],
    }

In [65]:
def evaluate_task_a_small_sample(
    n: int = 10,
    provider: str = LLM_PROVIDER,
    random_state: int = 123,
    locale_hint: Optional[str] = None,
    sleep_s: float = 5.0,
    force_profile_refresh: bool = False,
) -> pd.DataFrame:
    """
    Very small iteration loop. Use this to compare prompt/provider variants quickly.

    This version keeps dataframe columns consistent even when a row fails.
    It also stores the error string for debugging.
    """
    rows = internal_val_df.sample(n, random_state=random_state).copy()
    records = []

    for _, row in tqdm(rows.iterrows(), total=len(rows)):
        base_record = {
            "user_id": row["user_id"],
            "domain": row["domain"],
            "parent_asin": row["parent_asin"],
            "true_rating": float(row["rating"]),
            "pred_rating": None,
            "abs_error": None,
            "sq_error": None,
            "review_title": row.get("review_title"),
            "review_text": row.get("review_text"),
            "pred_review_title": None,
            "pred_review_text": None,
            "pred_rationale": None,
            "rating_confidence": None,
            "style_match_notes": None,
            "possible_failure_modes": None,
            "provider": provider,
            "model": None,
            "attempts": None,
            "repaired": None,
            "error": None,
        }

        try:
            pred = predict_task_a(
                user_id=row["user_id"],
                target_domain=row["domain"],
                target_parent_asin=row["parent_asin"],
                provider=provider,
                locale_hint=locale_hint,
                force_profile_refresh=force_profile_refresh,
            )

            y_true = float(row["rating"])
            y_pred = pred["prediction"].get("predicted_rating")
            err = None if y_pred is None else float(y_pred) - y_true

            base_record.update({
                "pred_rating": y_pred,
                "abs_error": None if err is None else abs(err),
                "sq_error": None if err is None else err ** 2,
                "pred_review_title": pred["prediction"].get("generated_review_title"),
                "pred_review_text": pred["prediction"].get("generated_review_text"),
                "pred_rationale": pred["prediction"].get("rating_rationale"),
                "rating_confidence": pred["prediction"].get("rating_confidence"),
                "style_match_notes": pred["prediction"].get("style_match_notes"),
                "possible_failure_modes": pred["prediction"].get("possible_failure_modes"),
                "model": pred.get("model"),
                "attempts": pred.get("attempts"),
                "repaired": pred.get("repaired"),
            })

        except Exception as e:
            base_record["error"] = repr(e)

        records.append(base_record)

        if sleep_s:
            time.sleep(sleep_s)

    df = pd.DataFrame(records)

    n_success = df["pred_rating"].notna().sum()
    n_failed = df["error"].notna().sum()

    if df["sq_error"].notna().any():
        rmse = math.sqrt(df["sq_error"].dropna().mean())
        mae = df["abs_error"].dropna().mean()
        print(f"success={n_success}/{len(df)} failed={n_failed}/{len(df)} RMSE={rmse:.4f} MAE={mae:.4f}")
    else:
        print(f"success={n_success}/{len(df)} failed={n_failed}/{len(df)}")

    return df


# Example:
results_df = evaluate_task_a_small_sample(
    n=20,
    provider="deepseek",
    random_state=10,
    sleep_s=5.0,
)

results_df

  0%|          | 0/20 [00:00<?, ?it/s]

success=18/20 failed=2/20 RMSE=0.6346 MAE=0.4167


,user_id,domain,parent_asin,true_rating,pred_rating,abs_error,sq_error,review_title,review_text,pred_review_title,pred_review_text,pred_rationale,rating_confidence,style_match_notes,possible_failure_modes,provider,model,attempts,repaired,error
0,AFECJFZZCGTND6HWT4EENHOVLYGA,Books,0439399297,5.0,5.0,0.0,0.00,Know more words before 1st grade.,Helpful,Five Stars,Great book!,"User is highly lenient with a default 5-star rating. No explicit drawbacks like high cost; price is low. No evidence in Books domain, but user's rating patt...",medium,"[Title is typical rating phrase, Extremely short review matching user's concise style, Enthusiastic but generic positive tone]","[Cross-domain prediction: user has never reviewed books, No detailed text to confirm fit with user's specific needs, Assumes user's leniency extends to educ...",deepseek,deepseek-v4-pro,1.0,False,None
1,AH7PVXZ74WJBCGMNFNOQRBYW6JBQ,Movies_and_TV,B001LRJH0U,5.0,4.0,1.0,1.00,Five Stars,GREAT WESTERN,Decent product with minor flaws,Overall a solid product. It meets expectations but could improve on durability.,"Based on similar user preferences and item characteristics, the predicted rating is estimated.",medium,"[Neutral tone, Concise]","[Cold start, Insufficient user history]",deepseek,deepseek-v4-pro,2.0,True,None
2,AGE54WMA3G553RWQ6XKDZFENKWDQ,Movies_and_TV,B00BNAE6M4,5.0,5.0,0.0,0.00,Five Stars,"I LOVED this movie, a must buy!",Five Stars,Loved this holiday romance! :),"Item is a holiday romantic movie, matching user's strong preference for heartwarming, family-friendly content. User gave 5 stars to similar Hallmark-style m...",high,"[Title uses star rating as typical, Review is very short (7 words), casual, and enthusiastic with a smiley, Tone matches user's brief positive expressions l...","[User's extreme leniency may overpredict 5, but metadata strongly aligns, No conflicting signals; unlikely to be rated lower]",deepseek,deepseek-v4-pro,1.0,False,None
3,AHZDVWFWT4QH4YENCWP75UJ3JGIQ,Books,B0030CVRXY,4.0,4.0,0.0,0.00,"A fast-moving narrative of World War II, featuring 3 vivid, powerful women characters.","Set in 1940, just before the U.S. entered the war, ""The Postmistress"" is a subtle, nuanced portrayal of the impact war has on three women: Frankie, a Britis...",A WWII tale of two women connected by war,"In 1940, postmistress Iris James holds the secrets of her small town, while radio journalist Frankie Bard reports from London. Their lives intersect through...","User typically rates historical fiction highly (mostly 4-5 stars), and this WWII novel fits their interest. However, without specific opinion, a 4.0 aligns ...",medium,"[Medium length, descriptive title with colon-like structure, Mentions slow start but positive overall, matching user's 4-star review style, Focuses on plot ...","[User may rate higher if they find it more captivating, or lower if they find it predictable, Sparse metadata from target item limits precise rating predict...",deepseek,deepseek-v4-pro,1.0,False,None
4,AEBJQFVDBGQDMTVA5BUNCLHCEZZA,Books,0486299759,5.0,NaN,NaN,NaN,Five Stars,Love the fairy stickers!!,None,None,None,None,None,None,deepseek,None,NaN,None,"ConnectionError(ReadTimeoutError(""HTTPSConnectionPool(host='api.deepseek.com', port=443): Read timed out.""))"
5,AH7CIIKURFCB26O2SKVCBDFFN3CA,Movies_and_TV,B0073U2F12,5.0,5.0,0.0,0.00,PERFECT 3D...A MUST FOR YOUR 3D COLLECTION!,PERFECT 3D!,SOLID SCI-FI ADDITION FOR YOUR HT 3D COLLECTION!,"The 3D on this one is super in my HT. If you enjoy alien invasion flicks, this is a worthy addition to your sci-fi collection. No issues with the disc - pic...",User defaults to 5 stars for sci-fi films unless a technical flaw is evident. The target is a sci-fi Blu-ray 3D that fits the user's collection and home the...,high,"[ALL CAPS enthusiastic title, Short, positive review mentioning HT and collection, Uses casual intensifiers like 'super' and 'top notch']","[Sparse metadata may hide technical details; defaulting to 5 stars is consistent

In [66]:
results_df.to_csv("task_a_small_sample_results_5_pro.csv", index=False)

In [59]:
target = internal_val_df.sample(1, random_state=7).iloc[0]

pred = predict_task_a(
    user_id=target["user_id"],
    target_domain=target["domain"],
    target_parent_asin=target["parent_asin"],
    provider="deepseek",  # or "gemini"
    locale_hint=None,
)

print("GROUND TRUTH RATING:", target["rating"])
print("ATTEMPTS:", pred["attempts"], "REPAIRED:", pred["repaired"])
print(json.dumps(pred["prediction"], indent=2, ensure_ascii=False))

GROUND TRUTH RATING: 5.0
ATTEMPTS: 1 REPAIRED: False
{
  "predicted_rating": 5.0,
  "rating_confidence": "medium",
  "rating_rationale": "User loves action movies and frequently gives 5 stars to favorites like Quentin Tarantino and Clint Eastwood collections. Independence Day is a classic action blockbuster with Will Smith, fitting user's preference for action and value. Price is low ($9.60), which user often highlights. No technical quality issues are known, so likely a positive rating aligned with user's lenient distribution (median 5.0).",
  "generated_review_title": "Five Stars",
  "generated_review_text": "Great action movie with Will Smith. Awesome special effects and a fun story. For the price, you can't beat it. If you like action and alien invasion flicks, this is a must-have.",
  "style_match_notes": [
    "Title is short and indicative of rating (consistent with 'Five Stars' used in past).",
    "Tone is positive and recommending, with direct language.",
    "Focus on value 

In [46]:
internal_val_df.head(2)

,domain,user_id,asin,parent_asin,rating,review_title,review_text,timestamp,verified_purchase,helpful_vote,rn,n_user_reviews,split
0,Movies_and_TV,AE225Z2VRWT6GPTOMA4H4O3H2KVQ,B0000E2R6P,B0000E2R6P,5.0,EXCELLENT old fashioned Horror,Yea Mr. Silva! EXCELLENT old fashioned Horror!!,1415579707000,True,0,31,40,internal_val
1,Books,AE225Z2VRWT6GPTOMA4H4O3H2KVQ,B005OH9A6E,B005OH9A6E,5.0,cool movie!,I haven't finished reading it yet...almost done...I am so gone! Taken away and right in there wit the characters! Dean Koontz has done it again...would make...,1416969499000,True,1,32,40,internal_val


In [ ]:
# Prompt iteration notes:
#
# Baseline to beat on internal validation rating:
#   user_item_bias RMSE ≈ 0.9577
#   user_mean      RMSE ≈ 0.9754
#
# The LLM may not beat these immediately on RMSE.
# Use this notebook to improve:
#   1. rating calibration
#   2. profile compactness
#   3. review tone/style fidelity
#
# Good next experiments:
#   - Add item average_rating/rating_number to target metadata when available.
#   - Add same-domain examples first if Task A target is same-domain.
#   - Add explicit calibration: "user mean is X; predict delta from mean".
#   - Generate rating only first, then review in a second call if rating/review mismatch occurs.